In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


## Load Data

In [ ]:
strava_gwr_features = gpd.read_file().to_crs(epsg=2193)

In [ ]:
feature_df = strava_gwr_features.drop(columns=['edgeUID', 'osm_reference_id', 'activity_class', 'source', 'geometry', 'buff_geometry'])
count_df = strava_gwr_features[['total_count']]

In [ ]:

def check_multicollinearity(df, feature_cols, vif_threshold=7.5):
    """
    Performs correlation and VIF checks on candidate independent variables for GWR.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the dataset.
    feature_cols : list of str
        List of predictor variable column names.
    vif_threshold : float
        Threshold for flagging high VIF values (typically 5 to 10 for GWR).
        
    Returns:
    --------
    vif_df : pandas.DataFrame
        DataFrame containing features and their corresponding VIF scores.
    """
    # Select feature subset and drop missing values
    X = df[feature_cols].copy().dropna()
    
    # -------------------------------------------------------------
    # 1. Pairwise Correlation Heatmap
    # -------------------------------------------------------------
    corr_matrix = X.corr()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Predictor Correlation Heatmap", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    
    # -------------------------------------------------------------
    # 2. Variance Inflation Factor (VIF)
    # -------------------------------------------------------------
    # Add constant term (intercept) required for accurate VIF calculation
    X_with_const = add_constant(X)
    
    vif_data = []
    for i, col in enumerate(X_with_const.columns):
        if col == "const":
            continue  # Skip intercept term in the final summary
            
        vif_val = variance_inflation_factor(X_with_const.values, i)
        vif_data.append({
            "Variable": col,
            "VIF": round(vif_val, 2),
            "Status": "High Collinearity" if vif_val >= vif_threshold else "OK"
        })
        
    vif_df = pd.DataFrame(vif_data).sort_values(by="VIF", ascending=False)
    
    print("\n================ VIF RESULTS ================")
    print(vif_df.to_string(index=False))
    print("=============================================\n")
    
    return vif_df

In [ ]:
vif_results = check_multicollinearity(count_df, feature_df, vif_threshold=7.5)